# PyQt5: QListWidget ve QListView — Kitap Listesi
### Görsel Programlama | Qt Designer → Python dönüşümü → PyCharm

**Hedef:** Aynı kitap ekleme/silme/temizleme uygulamasını iki farklı widget ile geliştirip veri yönetimi farkını kavramak.

Bu notebook, paketteki **gerçek `.ui` ve `.py` dosyalarını** açıklamaktadır. Kodlar masaüstü uygulaması olduğu için PyCharm terminalinden çalıştırılır; notebook hücreleri öğretim ve inceleme amaçlıdır. `uic.loadUi()` kullanılmaz.

**Öğrenme çıktıları:** Widget ve model kavramını ayırt etme; `pyuic` ile arayüz dönüştürme; sinyal-slot bağlantısı; seçim kontrolü; `QListWidget` ve `QListView` farkı.

## 1. Ön bilgi: İki yaklaşım neden var?

- **QListWidget:** Öğeleri `addItem()`/`addItems()` ile doğrudan widget'a ekleriz. Kendi dahili modeli vardır; başlangıç düzeyinde model sınıfı oluşturmaya gerek yoktur.
- **QListView:** Görüntüleme bileşenidir. Bir model bağlanmadan kitap listesini kendiliğinden yönetmez. Bu örnekte model `QStringListModel`.

```text
QListWidget                         QListView
───────────                         ─────────
addItem('Kitap')                    Python listesi
       │                                  │
       ▼                                  ▼
Widget'ın dahili modeli            QStringListModel
       │                                  │ setModel(...)
       ▼                                  ▼
Ekranda kitap                      QListView → Ekranda kitap
```

**Önemli ayrıntı:** Her ikisi de Qt Model/View altyapısından yararlanır. Fark, `QListWidget`'ın kolaylık sağlayan hazır öğe arayüzü sunması; `QListView`'ın modelin açıkça verilmesini beklemesidir.

## 2. Qt Designer'da iki arayüz

Her iki `.ui` dosyasında da şu `objectName` değerleri bulunur:

| Bileşen | Adı | Görevi |
|---|---|---|
| `QLineEdit` | `txtKitap` | Kitap adını yazmak |
| `QPushButton` | `btnEkle` | Kitap eklemek |
| `QPushButton` | `btnSil` | Seçili kitabı silmek |
| `QPushButton` | `btnTemizle` | Listeyi boşaltmak |
| `QLabel` | `lblDurum` | Toplam kitap sayısını göstermek |
| `QListWidget` **veya** `QListView` | `listKitaplar` | Kitapları göstermek |

Qt Designer'da dosyaları açıp **Widget Box**, **Object Inspector** ve **Property Editor** üzerinden tür ve `objectName` değerlerini kontrol edin.

**Beklenen görünüm (şematik):**

```text
┌────────────── Kitap Listesi ──────────────┐
│ Kitap adı: [___________________________]  │
│ [Ekle]  [Seçileni Sil]  [Temizle]         │
│ ┌──────────────────────────────────────┐  │
│ │ Python Temelleri                     │  │
│ │ Algoritmalar                         │  │
│ └──────────────────────────────────────┘  │
│ Toplam kitap: 2                          │
└──────────────────────────────────────────┘
```

İki örneğin arayüzü benzer görünür; asıl fark Python kodundadır.

## 3. `.ui` dosyasını Python'a dönüştürmek

PyCharm terminalinde **ilgili dosyaların bulunduğu klasörde** aşağıdaki komutları çalıştırın. PyQt5 yüklü olmalıdır (`python -m pip install PyQt5`).

**QListWidget:**
```bash
python -m PyQt5.uic.pyuic kitap_listwidget.ui -o ui_kitap_listwidget.py
python kitap_listwidget.py
```

**QListView:**
```bash
python -m PyQt5.uic.pyuic liste_gorunumu.ui -o ui_liste_gorunumu.py
python liste_gorunumu.py
```

Dönüşüm sonunda `ui_kitap_listwidget.py` / `ui_liste_gorunumu.py` otomatik oluşur. Bunlar **arayüzü oluşturan kodlardır**; olay mantığını kendi `kitap_listwidget.py` / `liste_gorunumu.py` dosyalarımızda yazarız.

```python
from ui_kitap_listwidget import Ui_MainWindow
class KitapPenceresi(QMainWindow, Ui_MainWindow):
    def __init__(self):
        super().__init__()
        self.setupUi(self)
```

`setupUi(self)`, Designer'da çizilen widget'ları pencere üzerinde kurar. `.ui` dosyasını sonradan değiştirirseniz **yeniden dönüştürmeniz gerekir**. Üretilen `ui_*.py` dosyasını elle düzenlemeyin.

## 4. QListWidget: İlk örnek

**Akış:** Kullanıcı kitap adını girer → `btnEkle.clicked` sinyali `ekle()` metodunu çağırır → `addItem()` ile öğe eklenir → `count()` ile durum etiketi güncellenir.

### 4.1 Başlangıç ve sinyaller

In [ ]:
# Bu parça kitap_listwidget.py dosyasındaki __init__ mantığıdır.
# PyCharm'da çalıştırılacak tam kod, paket içindeki kitap_listwidget.py dosyasındadır.
self.setupUi(self)
self.listKitaplar.addItems(["Python Temelleri", "Algoritmalar"])
self.btnEkle.clicked.connect(self.ekle)
self.btnSil.clicked.connect(self.sil)
self.btnTemizle.clicked.connect(self.temizle)
self.txtKitap.returnPressed.connect(self.ekle)
self.sayiyi_yaz()

`clicked.connect(...)`: Bir butonun **sinyalini**, çalıştırılacak **metoda (slot)** bağlar. `returnPressed`, metin kutusunda Enter'a basılınca tetiklenir.

### 4.2 Kitap ekleme

In [ ]:
def ekle(self):
    # Kullanıcının yazdığı metni al, gereksiz boşlukları sil.
    kitap = self.txtKitap.text().strip()
    # Boş girişte işlemi sonlandır.
    if not kitap:
        QMessageBox.warning(self, "Eksik bilgi", "Kitap adı giriniz.")
        return
    # Model sınıfı oluşturmadan doğrudan QListWidget'a ekle.
    self.listKitaplar.addItem(kitap)
    self.txtKitap.clear()
    self.txtKitap.setFocus()
    self.sayiyi_yaz()

### 4.3 Seçili kitabı silme ve sayıyı güncelleme

In [ ]:
def sil(self):
    satir = self.listKitaplar.currentRow()  # Seçili satırın indeksi
    if satir == -1:                         # Seçim yapılmadı
        QMessageBox.information(self, "Seçim", "Önce bir kitap seçiniz.")
        return
    self.listKitaplar.takeItem(satir)       # Satırı listeden çıkar
    self.sayiyi_yaz()

def sayiyi_yaz(self):
    adet = self.listKitaplar.count()
    self.lblDurum.setText(f"Toplam kitap: {adet}")

### 4.4 Tüm listeyi temizleme

In [ ]:
def temizle(self):
    if self.listKitaplar.count() == 0:
        QMessageBox.information(self, "Bilgi", "Liste zaten boş.")
        return
    self.listKitaplar.clear()
    self.sayiyi_yaz()

### 4.5 Tam kaynak kod: QListWidget

Aşağıdaki hücre paketteki `kitap_listwidget.py` dosyasının **birebir içeriğidir**. PyCharm'a kopyalayabilir veya dosyayı doğrudan açabilirsiniz.

In [ ]:
"""QListWidget örneği: Kitap listesi.

1) Arayüzü Python dosyasına dönüştür:
   python -m PyQt5.uic.pyuic kitap_listwidget.ui -o ui_kitap_listwidget.py
2) Uygulamayı çalıştır:
   python kitap_listwidget.py

NOT: Bu örnekte uic.loadUi() KULLANILMAZ.
"""

import sys
from PyQt5.QtWidgets import QApplication, QMainWindow, QMessageBox

# Qt Designer'ın .ui dosyasından ürettiğimiz Python sınıfını içe aktarıyoruz.
# Bu dosya, yukarıdaki dönüştürme komutuyla oluşur.
from ui_kitap_listwidget import Ui_MainWindow


class KitapPenceresi(QMainWindow, Ui_MainWindow):
    """Ana pencere sınıfı: QMainWindow ve üretilmiş arayüz sınıfını miras alır."""

    def __init__(self):
        super().__init__()

        # Designer'da çizilen widget'ları bu pencere üzerinde oluşturur.
        # Artık self.txtKitap, self.listKitaplar gibi nesnelere erişebiliriz.
        self.setupUi(self)

        # QListWidget, QListView'dan farklı olarak ayrı bir
        # QStringListModel oluşturulmasını GEREKTİRMEZ.
        # addItems() ile başlangıç kitaplarını doğrudan ekliyoruz.
        self.listKitaplar.addItems(["Python Temelleri", "Algoritmalar"])

        # Butonların clicked sinyalini ilgili fonksiyonlara bağlıyoruz.
        self.btnEkle.clicked.connect(self.ekle)
        self.btnSil.clicked.connect(self.sil)
        self.btnTemizle.clicked.connect(self.temizle)

        # Kullanıcı kitap adını yazıp Enter'a basarsa da ekleme yapılır.
        self.txtKitap.returnPressed.connect(self.ekle)

        # Başlangıçtaki kitap sayısını etikete yazdırıyoruz.
        self.sayiyi_yaz()

    def sayiyi_yaz(self):
        """QListWidget içindeki mevcut eleman sayısını gösterir."""
        # count() widget içindeki toplam satır/öğe sayısını döndürür.
        adet = self.listKitaplar.count()
        self.lblDurum.setText(f"Toplam kitap: {adet}")

    def ekle(self):
        """Metin kutusundaki kitabı listeye ekler."""
        # text() metni alır; strip() baştaki ve sondaki boşlukları temizler.
        kitap = self.txtKitap.text().strip()

        # Boş bir kitap adı eklenmesini önlüyoruz.
        if not kitap:
            QMessageBox.warning(self, "Eksik bilgi", "Kitap adı giriniz.")
            return

        # QListWidget'a doğrudan bir öğe ekliyoruz.
        # QListView örneğindeki gibi model.setStringList() gerekmez.
        self.listKitaplar.addItem(kitap)

        # Giriş alanını temizleyip imleci tekrar oraya taşıyoruz.
        self.txtKitap.clear()
        self.txtKitap.setFocus()

        # Kitap sayısı değiştiği için etiketi güncelliyoruz.
        self.sayiyi_yaz()

    def sil(self):
        """Kullanıcının seçtiği kitabı listeden kaldırır."""
        # currentRow() seçili öğenin sıra numarasını verir (0'dan başlar).
        # Hiçbir öğe seçili değilse -1 döndürür.
        satir = self.listKitaplar.currentRow()

        if satir == -1:
            QMessageBox.information(self, "Seçim", "Önce bir kitap seçiniz.")
            return

        # takeItem(satir) seçili satırdaki öğeyi listeden çıkarır.
        self.listKitaplar.takeItem(satir)
        self.sayiyi_yaz()

    def temizle(self):
        """Listedeki bütün kitapları kaldırır."""
        # Liste zaten boşsa işlem yapmaya gerek yok.
        if self.listKitaplar.count() == 0:
            QMessageBox.information(self, "Bilgi", "Liste zaten boş.")
            return

        # clear() listedeki tüm öğeleri siler.
        self.listKitaplar.clear()
        self.sayiyi_yaz()


if __name__ == "__main__":
    # Her PyQt masaüstü uygulamasında QApplication oluşturulur.
    app = QApplication(sys.argv)

    # Ana pencere nesnesini oluşturuyoruz.
    pencere = KitapPenceresi()
    pencere.show()

    # Qt olay döngüsünü başlatıyoruz.
    sys.exit(app.exec_())


## 5. QListView + QStringListModel

**Akış:** Kitaplar Python listesinde tutulur → `QStringListModel` oluşturulur → `listKitaplar.setModel(model)` ile görünüm modele bağlanır.

### 5.1 Modeli oluşturma

In [ ]:
self.kitaplar = ["Python Temelleri", "Algoritmalar"]  # Python listesi
self.model = QStringListModel(self.kitaplar)  # Qt model nesnesi
self.listKitaplar.setModel(self.model)        # Görünüm-model bağlantısı

### 5.2 Neden `modeli_yenile()` var?

Bu kaynak örnekte ekleme ve silme **Python listesi üzerinde** yapılır. `QStringListModel` bu listeyle otomatik olarak sürekli eşitlenmez. Bu yüzden her değişiklikten sonra `setStringList()` çağırıyoruz.

In [ ]:
def modeli_yenile(self):
    self.model.setStringList(self.kitaplar)
    self.sayiyi_yaz()

def sayiyi_yaz(self):
    self.lblDurum.setText(f"Toplam kitap: {len(self.kitaplar)}")

### 5.3 Ekleme, silme ve temizleme

In [ ]:
def ekle(self):
    kitap = self.txtKitap.text().strip()
    if not kitap:
        QMessageBox.warning(self, "Eksik bilgi", "Kitap adı giriniz.")
        return
    self.kitaplar.append(kitap)    # QListWidget'taki addItem() yerine
    self.modeli_yenile()          # Modeli yeni listeyle güncelle
    self.txtKitap.clear()
    self.txtKitap.setFocus()

def sil(self):
    index = self.listKitaplar.currentIndex()  # QModelIndex nesnesi
    if not index.isValid():
        QMessageBox.information(self, "Seçim", "Önce bir kitap seçiniz.")
        return
    self.kitaplar.pop(index.row())
    self.modeli_yenile()

def temizle(self):
    self.kitaplar.clear()
    self.modeli_yenile()

### 5.4 Tam kaynak kod: QListView

Aşağıdaki hücre, önceki ders paketindeki `liste_gorunumu.py` dosyasının **birebir içeriğidir**.

In [ ]:
"""QListView örneği: veriyi model tutar, QListView gösterir.
ÖNCE: python -m PyQt5.uic.pyuic liste_gorunumu.ui -o ui_liste_gorunumu.py
SONRA: python liste_gorunumu.py
"""
import sys
from PyQt5.QtCore import QStringListModel
from PyQt5.QtWidgets import QApplication, QMainWindow, QMessageBox
from ui_liste_gorunumu import Ui_MainWindow  # Qt Designer'dan üretilen Python sınıfı

class KitapPenceresi(QMainWindow, Ui_MainWindow):
    def __init__(self):
        super().__init__()
        self.setupUi(self)  # loadUi yok: üretilen sınıfın arayüzünü kurar

        # QListWidget'tan farklı olarak QListView bir MODEL ile çalışır.
        self.kitaplar = ["Python Temelleri", "Algoritmalar"]
        self.model = QStringListModel(self.kitaplar)
        self.listKitaplar.setModel(self.model)

        # Kullanıcı olaylarını fonksiyonlara bağla.
        self.btnEkle.clicked.connect(self.ekle)
        self.btnSil.clicked.connect(self.sil)
        self.btnTemizle.clicked.connect(self.temizle)
        self.txtKitap.returnPressed.connect(self.ekle)
        self.sayiyi_yaz()

    def modeli_yenile(self):
        # Liste değiştiğinde modelin içeriğini güncelliyoruz.
        self.model.setStringList(self.kitaplar)
        self.sayiyi_yaz()

    def sayiyi_yaz(self):
        self.lblDurum.setText(f"Toplam kitap: {len(self.kitaplar)}")

    def ekle(self):
        kitap = self.txtKitap.text().strip()
        if not kitap:
            QMessageBox.warning(self, "Eksik bilgi", "Kitap adı giriniz.")
            return
        self.kitaplar.append(kitap)
        self.modeli_yenile()
        self.txtKitap.clear()
        self.txtKitap.setFocus()

    def sil(self):
        # currentIndex() seçili öğenin QModelIndex bilgisini verir.
        index = self.listKitaplar.currentIndex()
        if not index.isValid():
            QMessageBox.information(self, "Seçim", "Önce bir kitap seçiniz.")
            return
        self.kitaplar.pop(index.row())
        self.modeli_yenile()

    def temizle(self):
        self.kitaplar.clear()
        self.modeli_yenile()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    pencere = KitapPenceresi()
    pencere.show()
    sys.exit(app.exec_())


## 6. Yan yana karşılaştırma

| İşlem | QListWidget | QListView + QStringListModel |
|---|---|---|
| İlk kitapları ekleme | `addItems([...])` | `QStringListModel([...])` + `setModel()` |
| Yeni kitap ekleme | `addItem(kitap)` | `kitaplar.append(kitap)` + `setStringList()` |
| Seçimi alma | `currentRow()` | `currentIndex()` |
| Seçim geçerli mi? | `satir != -1` | `index.isValid()` |
| Kitap silme | `takeItem(satir)` | `kitaplar.pop(index.row())` + modeli yenile |
| Listeyi boşaltma | `clear()` | `kitaplar.clear()` + modeli yenile |
| Eleman sayısı | `count()` | `len(kitaplar)` |

**Kavramsal uyarı:** `QListView`'da her zaman ayrı Python listesi tutmak zorunda değilsiniz; burada ders örneğinin tercih ettiği yöntem kullanılmıştır. Daha ileri düzeyde modelin kendisi üzerinden veri düzenlenebilir.

## 7. Çalışma senaryoları ve beklenen sonuçlar

| Adım | Kullanıcı eylemi | Beklenen sonuç |
|---|---|---|
| 1 | Programı aç | İki kitap; “Toplam kitap: 2” |
| 2 | “Veri Yapıları” yaz ve Ekle | Üç kitap; “Toplam kitap: 3” |
| 3 | Boş metinle Ekle | Uyarı; liste değişmez |
| 4 | Bir kitabı seç ve Sil | Seçili kitap çıkar; sayı azalır |
| 5 | Hiç seçim yokken Sil | Bilgilendirme mesajı |
| 6 | Temizle | Liste boşalır; sayı 0 olur |
| 7 | Kitap adını yazıp Enter'a bas | Kitap eklenir |

**Sınıfta tartışın:** Ekrandaki davranış aynıyken veri yönetimindeki sorumluluklar nasıl değişiyor?

## 8. Sıra Sizde 1 — Aynı kitabı iki kez eklemeyi engelleyin

**Görev:** Kullanıcı “Algoritmalar” kitabını yeniden eklemek isterse `QMessageBox.information()` gösterilsin ve yeni öğe eklenmesin. Hem `QListWidget` hem `QListView` sürümünü düşünün.

**İpucu:** `QListWidget` için `count()` ve `item(i).text()`; `QListView` için `self.kitaplar` kullanılabilir.

<details><summary>Çözümü Göster — QListWidget</summary>

```python
# ekle() metodunda, addItem() satırından ÖNCE:
for i in range(self.listKitaplar.count()):
    if self.listKitaplar.item(i).text().casefold() == kitap.casefold():
        QMessageBox.information(self, "Tekrar", "Bu kitap zaten var.")
        return
```
</details>

<details><summary>Çözümü Göster — QListView</summary>

```python
# ekle() metodunda, append() satırından ÖNCE:
if kitap.casefold() in [k.casefold() for k in self.kitaplar]:
    QMessageBox.information(self, "Tekrar", "Bu kitap zaten var.")
    return
```
</details>

## 9. Sıra Sizde 2 — Seçili kitabı etiket üzerinde gösterin

**Görev:** Listedeki bir kitaba tıklanınca `lblDurum` üzerinde seçili kitabın adı da yazsın. Bunun için `clicked` sinyalini kullanabilirsiniz.

<details><summary>Çözümü Göster — QListWidget</summary>

```python
# __init__ içinde:
self.listKitaplar.itemClicked.connect(self.secimi_goster)

# Sınıfa yeni metot:
def secimi_goster(self, item):
    self.lblDurum.setText(f"Seçili: {item.text()} | Toplam: {self.listKitaplar.count()}")
```
</details>

<details><summary>Çözümü Göster — QListView</summary>

```python
# __init__ içinde:
self.listKitaplar.clicked.connect(self.secimi_goster)

# Sınıfa yeni metot:
def secimi_goster(self, index):
    self.lblDurum.setText(f"Seçili: {index.data()} | Toplam: {len(self.kitaplar)}")
```
</details>

**Not:** Ekleme/silme sonrası mevcut `sayiyi_yaz()` fonksiyonu etiketi yeniden yalnızca toplam sayıya döndürür. Bu, mevcut örneğin davranışıdır.

## 10. Sıra Sizde 3 — Kitap adını güncelleyin

**Görev:** Seçili kitabın adını `txtKitap` kutusundaki metinle değiştirin. Qt Designer'a `btnGuncelle` adında yeni bir buton ekleyin, `.ui` dosyasını yeniden Python'a dönüştürün ve `clicked.connect()` ile metodu bağlayın.

<details><summary>Çözümü Göster — QListWidget</summary>

```python
# __init__ içine:
self.btnGuncelle.clicked.connect(self.guncelle)

def guncelle(self):
    item = self.listKitaplar.currentItem()
    yeni = self.txtKitap.text().strip()
    if item is None or not yeni:
        QMessageBox.warning(self, "Uyarı", "Kitap seçip yeni ad yazınız.")
        return
    item.setText(yeni)
    self.txtKitap.clear()
```
</details>

<details><summary>Çözümü Göster — QListView</summary>

```python
# __init__ içine:
self.btnGuncelle.clicked.connect(self.guncelle)

def guncelle(self):
    index = self.listKitaplar.currentIndex()
    yeni = self.txtKitap.text().strip()
    if not index.isValid() or not yeni:
        QMessageBox.warning(self, "Uyarı", "Kitap seçip yeni ad yazınız.")
        return
    self.kitaplar[index.row()] = yeni
    self.modeli_yenile()
    self.txtKitap.clear()
```
</details>

## 11. Hata bulma etkinliği

**Hata A:** `ModuleNotFoundError: No module named 'ui_liste_gorunumu'` — nedeni nedir?

**Hata B:** `AttributeError: 'KitapPenceresi' object has no attribute 'listKitaplar'` — neresi kontrol edilir?

**Hata C:** `QListView` sürümünde `self.kitaplar.append(kitap)` çalışıyor ama ekranda kitap görünmüyor — hangi çağrı eksik?

<details><summary>Çözümleri Göster</summary>

- **A:** `.ui` dosyası henüz `pyuic` ile dönüştürülmemiş olabilir veya üretilen dosya aynı klasörde değildir.
- **B:** Designer'daki `objectName` ve dönüştürülmüş `ui_*.py` dosyasının güncel olup olmadığı kontrol edilir.
- **C:** `self.modeli_yenile()` / `self.model.setStringList(self.kitaplar)` çağrısı eksiktir.
</details>

## 12. Mini değerlendirme

1. `QListWidget` neden ayrıca `QStringListModel` gerektirmiyor?
2. `currentRow()` ile `currentIndex()` ne döndürür?
3. `index.isValid()` kontrolü neden önemlidir?
4. `setupUi(self)` hangi dosyada tanımlanan sınıftan gelir?
5. `.ui` dosyasında yeni buton eklediğinizde neden `pyuic` tekrar çalıştırılır?

<details><summary>Örnek Yanıtlar</summary>

1. Öğeleri yönetmek için kendi dahili modeline ve kolaylık metotlarına sahiptir.
2. İlki satır numarası (`int`), ikincisi `QModelIndex` nesnesi döndürür.
3. Geçerli seçim yokken hatalı satır silme girişimini önler.
4. `.ui` dosyasından üretilen `Ui_MainWindow` sınıfından.
5. Arayüzün Python karşılığı yeni tasarımı yansıtmalıdır.
</details>

**Sonraki konu önerisi:** `QTableWidget` ve `QTableView` karşılaştırması.